In [1]:
from moe_reft.olmoe.modeling_olmoe import OlmoeForCausalLM
from moe_reft.olmoe import configuration_olmoe
from moe_reft import interventions_config

model = OlmoeForCausalLM(configuration_olmoe.OlmoeInterventionsConfig(
    interventions_config=interventions_config.InterventionsConfig(
        intervention_places="after_moe",
        intervention_layers="even_only",
    ),
))

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
from __future__ import annotations

import torch
from loguru import logger
from torch import nn
from transformers import AutoModelForCausalLM, PreTrainedModel

map_dtype=torch.bfloat16  # optional casting
map_device=torch.device("cuda")  # optional device move
        
hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct"
# hf_model_name_or_path="allenai/OLMoE-1B-7B-0924-Instruct"

hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
        hf_model_name_or_path,
        dtype=map_dtype if map_dtype is not None else None,
        trust_remote_code=True,
    )

src_sd = hf_model.state_dict()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
from moe_reft.olmoe import load_weights

intervention_patterns = [
    "*.pre_moe_intervention.*",
    "*.after_moe_intervention.*",
    "*.pre_moe_intervenetion.*",  # typo fallback
]
    # 1) Load HF model & grab its state dict
hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    hf_model_name_or_path,
    dtype=map_dtype if map_dtype is not None else None,
    trust_remote_code="True",
)
src_sd = hf_model.state_dict()

# 2) Build filtered state dict compatible with your custom model
filtered_sd, report = load_weights.build_partial_state_dict(
    src_sd=src_sd,
    dst_module=model,
    intervention_patterns=intervention_patterns,
    device=map_device,
    dtype=map_dtype,
)

# 3) Load with strict=False (so missing keys — e.g., interventions — are fine)
missing, unexpected = model.load_state_dict(filtered_sd, strict=False)

# Merge loader feedback into the report
report.skipped_missing.extend(missing)
if unexpected:
    report.skipped_missing.extend(unexpected)

# 4) Freeze everything, then unfreeze only intervention layers
for param in model.parameters():
    param.requires_grad = False


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Building partial state dict: 100%|██████████| 3219/3219 [00:03<00:00, 1049.63it/s]


In [6]:
for name, param in model.named_parameters():
    if load_weights._matches_any(name, intervention_patterns):
        param.requires_grad = True

# 5) Print parameter stats
total_params, trainable_params = load_weights._count_parameters(model)
print(f"Total parameters:     {total_params}")
print(f"Trainable parameters: {trainable_params}")

logger.info(f"Parameter stats — total: {total_params}, trainable: {trainable_params}")

2025-11-08 18:03:22.956 | INFO     | __main__:<module>:10 - Parameter stats — total: 6919424064, trainable: 262208


Total parameters:     6919424064
Trainable parameters: 262208


In [10]:
for name,param in model.named_parameters():
    if load_weights._matches_any(name, intervention_patterns):
        print(name, param.shape)

model.layers.0.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.0.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.0.after_moe_intervention.learned_source.bias torch.Size([8])
model.layers.2.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.2.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.2.after_moe_intervention.learned_source.bias torch.Size([8])
model.layers.4.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.4.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.4.after_moe_intervention.learned_source.bias torch.Size([8])
model.layers.6.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.6.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.6.after_moe_i